## Fine Tuning

In [5]:
#Open API Fine Tuning Dashboard URL: https://platform.openai.com/finetune/

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import os

# --- Load API Key ---
load_dotenv(override=True, dotenv_path="../.env")
my_api_key = os.getenv("OPENAI_API_KEY")


client = OpenAI(api_key=my_api_key)

In [6]:
with open("data/training_data.jsonl", "rb") as f:
    training_file_object = client.files.create(file=f, purpose = "fine-tune")

print("Uploaded training file ID:", training_file_object.id )

with open("data/validation_data.jsonl", "rb") as f:
    validation_file_object = client.files.create(file= f, purpose = "fine-tune")

print("Uploaded validation file ID:", validation_file_object.id )

Uploaded training file ID: file-A2yV7zDhUJ7M9xqB1vsKKJ
Uploaded validation file ID: file-Gkeqmu1FBM61LsD4E3iqCV


In [8]:
## Create Fine-Tune Job
job = client.fine_tuning.jobs.create(
    training_file = training_file_object.id,
    validation_file = validation_file_object.id,
    model = "gpt-4.1-nano-2025-04-14",
    suffix = "brand-customer-support"
)
print("Created Fine-Tune Job ID:", job.id)


Created Fine-Tune Job ID: ftjob-uM9dqK4o6KrUh70P3I6hkk66


In [10]:
jobs = client.fine_tuning.jobs.list(limit=5)
for j in jobs.data:
    print(j.id, j.status, j.fine_tuned_model)

ftjob-uM9dqK4o6KrUh70P3I6hkk66 running None
ftjob-oAgylwyBF69b2fAlYikW4ytH running None


In [16]:
def ask_question_without_finetuning(prompt):
    print(f"User asked: {prompt}")
    response = client.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Answer as concisely as possible."},
            {"role": "user", "content": prompt}
        ]
    )
    print (response)
    return response.choices[0].message.content  

def ask_question_with_finetuning(prompt):
    print(f"User asked: {prompt}")
    response = client.chat.completions.create(
        model="ft:gpt-4.1-nano-2025-04-14:personal:brand-customer-support:Cjr8t1XY",
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Answer as concisely as possible."},
            {"role": "user", "content": prompt}
        ]
    )
    print (response)
    return response.choices[0].message.content  

In [17]:
import time
while True:
    # Ask user for a question
    user_prompt = input("Ask something: ")

    if (user_prompt.lower() != 'quit'):
        # Get and print the response
        response = ask_question_without_finetuning(user_prompt)
        print("\n[WITHOUT FINE TUNING] OpenAI says:", response)

        response = ask_question_with_finetuning(user_prompt)
        print("\n[WITH FINE TUNING] OpenAI says:", response)

        # add delay of 3 seconds
        time.sleep(3)
    else:
        break    

User asked: what is the refund policy
ChatCompletion(id='chatcmpl-CjrQT2w5ieHhSu8X3v2aPT68vLrRE', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Policies vary by retailer. If you tell me the product/website or share a link, I can pull the exact terms. Here’s a quick generic guide:\n\n- Window: typically 7–30 days from delivery/purchase.\n- Condition: items usually must be unused/original packaging; digital goods are often non-refundable.\n- Proof: receipt/order number, payment method.\n- How to start: use the retailer’s returns portal or contact customer support; you may need to print a return label.\n- Fees: possible restocking or shipping fees; some items are non-refundable.\n- Refund method/timing: to the original payment method; processing usually 5–14 business days after the item is received.\n- Exclusions: final sale, customized items, opened software, subscriptions, etc.\n\nIf you share the retailer or provide a link, I can g